# Keras 3 — quick start (classification)

**Standalone Kaggle notebook** — minimal end-to-end pipeline: **Fashion-MNIST**, a small CNN, metrics and prediction plots.

Turn on a **GPU** accelerator on Kaggle for faster training.

In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

%matplotlib inline

print("Keras version:", keras.__version__)
print("Backend:", keras.config.backend())


In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)
num_classes = 10


## Sample images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(11, 4.5))
for ax, img, lab in zip(axes.ravel(), x_train[:10], y_train[:10]):
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(class_names[int(lab)])
    ax.axis("off")
plt.suptitle("First 10 training images")
plt.tight_layout()
plt.show()


In [ ]:
model = keras.Sequential(
    [
        keras.layers.Input(shape=(28, 28, 1)),
        keras.layers.Conv2D(32, 3, activation="relu"),
        keras.layers.MaxPooling2D(),
        keras.layers.Conv2D(64, 3, activation="relu"),
        keras.layers.MaxPooling2D(),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(num_classes, activation="softmax"),
    ]
)
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"],
)
model.summary()


In [ ]:
history = model.fit(
    x_train,
    y_train,
    epochs=12,
    batch_size=128,
    validation_split=0.1,
    verbose=1,
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.history["sparse_categorical_accuracy"], label="train")
axes[0].plot(history.history["val_sparse_categorical_accuracy"], label="val")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("epoch")
axes[0].legend()
axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="val")
axes[1].set_title("Loss")
axes[1].set_xlabel("epoch")
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}  |  loss: {test_loss:.4f}")

probs = model.predict(x_test, verbose=0)
y_pred = np.argmax(probs, axis=1)
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(cm, cmap="magma")
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticklabels(class_names)
ax.set_ylabel("True")
ax.set_xlabel("Predicted")
ax.set_title("Confusion matrix — Fashion-MNIST test")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=class_names))


## Where the model is most confused

In [ ]:
wrong = np.where(y_pred != y_test)[0]
rng = np.random.default_rng(1)
sample = rng.choice(wrong, size=min(10, len(wrong)), replace=False)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, i in zip(axes.ravel(), sample):
    ax.imshow(x_test[i].squeeze(), cmap="gray")
    ax.axis("off")
    ax.set_title(
        f"true: {class_names[y_test[i]]}\npred: {class_names[y_pred[i]]}",
        fontsize=7,
    )
plt.suptitle("Random misclassified test examples")
plt.tight_layout()
plt.show()
